이번 대회에서는 범주형 변수 전처리를 위해 Label Encoding과 for문을 사용했습니다.  

이는 train data로 fit한 Label Encoder로 test data를 transform할 경우,  

train data에는 속하지 않은 데이터가 test data에 있을 가능성이 있어 에러가 발생할 수 있기 때문입니다.  

이를 방지하기 위해 for문을 사용하여 예외적인 상황에 대처할 수 있도록 코드를 작성했습니다.  

참고해 주시길 바랍니다.  

# import / 라이브러리 호출

In [1]:
import pandas as pd
import random
import os
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeRegressor

# Fixed RandomSeed / 랜덤시드 고정
매번 고정된 결과를 얻기 위해서 사용합니다.  

seed를 고정하지 않는다면 같은 코드라도 매번 다른 결과가 나오게됩니다.  

항상 동일한 결과를 얻기 위해서 사용합니다.

In [2]:
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(42) # Seed 고정

# Data Load / 데이터 불러오기

In [3]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [4]:
train.head()

,ID,Exercise_Duration,Body_Temperature(F),BPM,Height(Feet),Height(Remainder_Inches),Weight(lb),Weight_Status,Gender,Age,Calories_Burned
0,TRAIN_0000,26.0,105.6,107.0,5.0,9.0,154.3,Normal Weight,F,45,166.0
1,TRAIN_0001,7.0,103.3,88.0,6.0,6.0,224.9,Overweight,M,50,33.0
2,TRAIN_0002,7.0,103.3,86.0,6.0,3.0,218.3,Overweight,M,29,23.0
3,TRAIN_0003,17.0,104.0,99.0,5.0,6.0,147.7,Normal Weight,F,33,91.0
4,TRAIN_0004,9.0,102.7,88.0,5.0,10.0,169.8,Normal Weight,M,38,32.0


# Feature & Target Split / 독립변수, 종속변수로 나누기  

모델을 학습시키기 위해서 독립변수(x)와 종속변수(y)로 나누어야 합니다.  

단, 분석에 활용될 수 없는 ID 데이터는 제거하겠습니다.

In [5]:
# 독립변수로 설정할 train_x에서는 종속변수를 제거합니다. 또한 분석에 활용하지 않는 ID 데이터를 제거합니다.
train_x = train.drop(['ID', 'Calories_Burned'], axis = 1)
# train_y 변수를 종속변수로 사용하기 위해 Calories_Burned 데이터를 지정하였습니다.
train_y = train['Calories_Burned']

# train_x 데이터와 마찬가지로 분석에 활용하지 않는 ID 데이터를 제거합니다.
test_x = test.drop('ID', axis = 1)

# Data Preprocessing / 데이터 전처리  

데이터 전처리는 결측치 제거, 데이터 단위 변환 등 데이터에 여러 가지 처리를 해주는 것입니다.  

전처리를 함으로써 데이터 분석이 가능하기 때문에 데이터 분석에 있어서 반드시 필수적인 부분입니다.  

이번 Baseline 코드에서 사용할 전처리 방법은 Label Encoding 입니다.

범주형 변수는 순차형 변수와 명목형 변수로 나눌 수 있습니다. 

명목형 변수는 순서 관계가 존재하지 않기 때문에 수치 레이블링(Labeling)으로는 관계를 제대로 표현할 수 없으나,  

순차형 변수의 경우 순서대로 수치값을 레이블로 부여하여 간단히 수치화 할 수 있습니다.   

따라서 Label Encoding 방법을 사용하여 문자열 데이터를 변환하겠습니다.

In [6]:
ordinal_features = ['Weight_Status', 'Gender']

for feature in ordinal_features:
    le = LabelEncoder()
    le = le.fit(train_x[feature])
    train_x[feature] = le.transform(train_x[feature])

    # train데이터에서 존재하지 않았던 값이 test 데이터에 존재할 수도 있습니다.
    # 따라서 test 데이터를 바로 변형시키지 않고 고윳값을 확인후 test 데이터를 변환합니다.
    for label in np.unique(test_x[feature]):
        if label not in le.classes_:
            le.classes_ = np.append(le.classes_, label)
    test_x[feature] = le.transform(test_x[feature])

# Regression Model Definition / 회귀 모델 정의

In [7]:
model = DecisionTreeRegressor(random_state = 42)

# Model fitting / 모델 학습

In [8]:
model.fit(train_x, train_y)

DecisionTreeRegressor(random_state=42)

# Inference / 추론

In [9]:
preds = model.predict(test_x)

# submit / 제출

In [10]:
submission = pd.read_csv('sample_submission.csv')

In [11]:
submission['Calories_Burned'] = preds

In [12]:
submission.to_csv('./submit.csv', index = False)